***D0. Load and preprocess (one pipeline only)***

1.Load sentences: brown.sents(categories='news')

In [8]:
import nltk
from nltk.corpus import brown

nltk.download("brown")

sents = brown.sents(categories="news")

print("Number of sentences:", len(sents))
print("Example sentence:", sents[0])

Number of sentences: 4623
Example sentence: ['The', 'Fulton', 'County', 'Grand', 'Jury', 'said', 'Friday', 'an', 'investigation', 'of', "Atlanta's", 'recent', 'primary', 'election', 'produced', '``', 'no', 'evidence', "''", 'that', 'any', 'irregularities', 'took', 'place', '.']


[nltk_data] Downloading package brown to /Users/luckgucci/nltk_data...
[nltk_data]   Package brown is already up-to-date!


2.Preprocess tokens:
lowercase
remove punctuation-only tokens (keep words like “don’t” as tokens)
keep stopwords (required for language modeling)


In [9]:
import re

_has_alnum = re.compile(r"[A-Za-z0-9]")

def preprocess_sentence(tokens):
    processed = []
    for t in tokens:
        t_lower = t.lower()

        if _has_alnum.search(t_lower) is None:
            continue
        
        processed.append(t_lower)
        
    return processed

sents_pp = [preprocess_sentence(sent) for sent in sents]

sents_pp = [sent for sent in sents_pp if len(sent) > 0]

print("Number of sentences after preprocessing:", len(sents_pp))
print("Example processed sentence:", sents_pp[0])

Number of sentences after preprocessing: 4611
Example processed sentence: ['the', 'fulton', 'county', 'grand', 'jury', 'said', 'friday', 'an', 'investigation', 'of', "atlanta's", 'recent', 'primary', 'election', 'produced', 'no', 'evidence', 'that', 'any', 'irregularities', 'took', 'place']


3.Add special tokens:
<bos> at the start of every sentence
<eos> at the end of every sentence
<unk> for unknown words (needed for PyTorch and for robust evaluation)


In [10]:
BOS = "<bos>"
EOS = "<eos>"
UNK = "<unk>"

sents_marked = [[BOS] + sent + [EOS] for sent in sents_pp]

print("Example sentence with special tokens:")
print(sents_marked[0])
print("\nCheck first 10 tokens:")
print(sents_marked[0][:10])

Example sentence with special tokens:
['<bos>', 'the', 'fulton', 'county', 'grand', 'jury', 'said', 'friday', 'an', 'investigation', 'of', "atlanta's", 'recent', 'primary', 'election', 'produced', 'no', 'evidence', 'that', 'any', 'irregularities', 'took', 'place', '<eos>']

Check first 10 tokens:
['<bos>', 'the', 'fulton', 'county', 'grand', 'jury', 'said', 'friday', 'an', 'investigation']


***D1. Split (same split used for both models)***

Split by sentence (not by token) to avoid leakage:
80% train
10% validation
10% test
Report:
number of sentences in each split
number of tokens in each split
vocabulary size (built from train only)


In [11]:
import random
from collections import Counter

def split_data(sentences, train_ratio=0.8, valid_ratio=0.1, seed=42):
    random.seed(seed)
    indices = list(range(len(sentences)))
    random.shuffle(indices)

    n = len(sentences)
    n_train = int(n * train_ratio)
    n_valid = int(n * valid_ratio)

    train_idx = indices[:n_train]
    valid_idx = indices[n_train:n_train+n_valid]
    test_idx  = indices[n_train+n_valid:]

    train = [sentences[i] for i in train_idx]
    valid = [sentences[i] for i in valid_idx]
    test  = [sentences[i] for i in test_idx]

    return train, valid, test

train_sents, valid_sents, test_sents = split_data(sents_marked)


counter = Counter()
for sent in train_sents:
    counter.update(sent)

# special tokens must exist
vocab = {BOS, EOS, UNK}

# add tokens from train
for word in counter:
    vocab.add(word)

vocab_size = len(vocab)

def replace_oov(sentences, vocab):
    new_data = []
    for sent in sentences:
        new_sent = [w if w in vocab else UNK for w in sent]
        new_data.append(new_sent)
    return new_data

valid_sents = replace_oov(valid_sents, vocab)
test_sents  = replace_oov(test_sents, vocab)

def count_tokens(sentences):
    return sum(len(sent) for sent in sentences)

train_tokens = count_tokens(train_sents)
valid_tokens = count_tokens(valid_sents)
test_tokens  = count_tokens(test_sents)

print("Sentence counts:")
print("Train:", len(train_sents))
print("Valid:", len(valid_sents))
print("Test :", len(test_sents))

print("\nToken counts:")
print("Train:", train_tokens)
print("Valid:", valid_tokens)
print("Test :", test_tokens)

print("\nVocabulary size (train only):", vocab_size)

Sentence counts:
Train: 3688
Valid: 461
Test : 462

Token counts:
Train: 77883
Valid: 10104
Test : 9827

Vocabulary size (train only): 11679


***D2. Vocabulary (shared)***

Build vocabulary from training split only with:
min_freq (e.g., 2 or 3)
all other words map to <unk>
Report:
chosen min_freq
final vocab size
top 20 most frequent tokens


In [13]:
from collections import Counter

min_freq = 2  

counter = Counter()
for sent in train_sents:
    counter.update(sent)

vocab = {BOS, EOS, UNK}

for word, freq in counter.items():
    if freq >= min_freq:
        vocab.add(word)

vocab_size = len(vocab)

def apply_unk(sentences, vocab):
    new_data = []
    for sent in sentences:
        new_sent = [w if w in vocab else UNK for w in sent]
        new_data.append(new_sent)
    return new_data

train_sents = apply_unk(train_sents, vocab)
valid_sents = apply_unk(valid_sents, vocab)
test_sents  = apply_unk(test_sents, vocab)

print("Chosen min_freq:", min_freq)
print("Final vocabulary size:", vocab_size)

print("\nTop 20 most frequent tokens (from train before UNK mapping):")
for word, freq in counter.most_common(20):
    print(word, ":", freq)

Chosen min_freq: 2
Final vocabulary size: 5396

Top 20 most frequent tokens (from train before UNK mapping):
the : 5073
<bos> : 3688
<eos> : 3688
of : 2288
and : 1734
a : 1690
to : 1682
in : 1581
for : 760
that : 653
is : 576
on : 556
was : 553
he : 517
at : 500
with : 471
as : 411
be : 403
by : 402
it : 379


***Part A – Statistical n-gram Language Model (OPTIONAL)***

A1. Train a Trigram Language Model
Using NLTK’s language modeling tools:
Train a trigram model:
You may use one of the following:
MLE (Maximum Likelihood Estimation)
Laplace (Add-1 smoothing)
Lidstone (Add-k smoothing)
Report:
Which model you used
Which smoothing method (if any)
Why smoothing is necessary


In [14]:
from nltk.lm import MLE, Laplace, Lidstone
from nltk.lm.preprocessing import padded_everygram_pipeline

n = 3
train_data, padded_vocab = padded_everygram_pipeline(n, train_sents)
model = Laplace(n)     
model.fit(train_data, padded_vocab)

print("Trigram model trained successfully.")

Trigram model trained successfully.


In [16]:
from nltk.lm.preprocessing import padded_everygram_pipeline
from itertools import chain

def compute_perplexity(model, sentences, n):
    test_data, _ = padded_everygram_pipeline(n, sentences)
    
    test_ngrams = list(chain.from_iterable(test_data))
    
    return model.perplexity(test_ngrams)

valid_ppl = compute_perplexity(model, valid_sents, n)
test_ppl  = compute_perplexity(model, test_sents, n)

print("Validation Perplexity:", valid_ppl)
print("Test Perplexity:", test_ppl)

Validation Perplexity: 553.1649311950218
Test Perplexity: 531.9683699497687


***We trained a trigram (n=3) language model using NLTK’s nltk.lm module with Laplace (Add-1) smoothing. Smoothing is necessary because, without it, unseen trigrams would receive probability 0, causing entire sentence probabilities to become 0 and resulting in infinite perplexity. Laplace smoothing assigns a small non-zero probability to unseen n-grams, enabling more robust evaluation. The model achieved a validation perplexity of 553.16 and a test perplexity of 531.97. Perplexity measures how well the model predicts unseen text, and lower values indicate better predictive performance.***

***A2. Evaluate with Perplexity***


***The trigram language model achieved a validation perplexity of 553.16 and a test perplexity of 531.97. These values are relatively high, which is expected for a trigram model trained on a moderate-sized corpus with a large vocabulary. Trigram models are limited because they only consider the previous two words as context, which restricts their ability to capture long-range dependencies. Several factors affect perplexity, including vocabulary size, the choice of smoothing method, the amount of training data, the value of n in the n-gram model, and the distributional differences between training and test data. Larger vocabularies and sparse data typically increase perplexity, while better smoothing and more training data generally reduce it.***

***A4. Text Generation***

Generate 3 short text samples (at least 30 tokens each).
Briefly comment:
Are they grammatical?
Are they coherent?
What limitations do you observe?


In [24]:
import random

def generate_text(model, max_tokens=40):
    generated = []
    context = ["<s>", "<s>"] 
    
    for _ in range(max_tokens):
        next_word = model.generate(text_seed=context[-2:])
        
        if next_word == "</s>":
            break
            
        generated.append(next_word)
        context.append(next_word)
    
    return " ".join(generated)

print("Sample 1:\n", generate_text(model, 50))
print("\nSample 2:\n", generate_text(model, 50))
print("\nSample 3:\n", generate_text(model, 50))

Sample 1:
 <bos> the brevard group took at the plant which this year <eos>

Sample 2:
 <bos> interested in setting up <unk> <eos>

Sample 3:
 <bos> her acting began with the best motel in norman okla. where the work was dull and <unk> cuts hospital authorities said <eos>


***The generated samples are partially grammatical at the local level, as many short word sequences follow plausible trigram patterns learned from the training data. However, the sentences are often incomplete and lack global coherence. Some outputs terminate early, and the presence of <unk> tokens indicates limited vocabulary coverage due to frequency filtering. Since a trigram model only conditions on the previous two words, it cannot capture long-range dependencies or maintain consistent sentence structure. As a result, while local word combinations may appear reasonable, the overall text often lacks meaningful structure and narrative flow.***